In [68]:
import pandas as pd
import numpy as np

Importing the libraries that will be used throughout the analysis.

In [70]:
df = pd.read_excel("Quiz-07.xlsx")

Loading the Excel dataset that will be used in the analysis and importing it into a DataFrame.Η ανάγνωση του excel πίνακα που θα χρησιμοποιηθεί, το οποίο εισάγεται σε ένα Dataframe

In [72]:
df.head(10)

,Passport Number,Date of Visit,Sales Amount,Transaction Number
0,10609000872612,2013-11-05,12.20,10001
1,10609000885845,2012-12-11,962.55,10001
2,10609000891732,2012-11-28,7.65,10001
3,10609001029507,2012-12-18,12.55,10001
4,10609001029507,2012-12-18,13.72,10001
5,10609001029507,2012-12-21,12.55,10001
6,10609001029507,2012-12-21,13.72,10001
7,10609001046605,2012-09-28,22.40,10001
8,10609001049099,2013-02-05,34.19,10001
9,10609001049099,2013-02-05,27.86,10001


An initial inspection of the dataset is performed in order to better understand the structure of the data and the available variables.

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18095 entries, 0 to 18094
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Passport Number     18095 non-null  int64         
 1   Date of Visit       18095 non-null  datetime64[ns]
 2   Sales Amount        18095 non-null  float64       
 3   Transaction Number  18095 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 565.6 KB


Before processing the data, the variable types are examined to ensure that each variable is properly defined for the analysis, particularly the datetime variables.

In [76]:
df.isnull().sum()

Passport Number       0
Date of Visit         0
Sales Amount          0
Transaction Number    0
dtype: int64

A check for missing values is then performed in order to determine whether any data cleaning procedures are required prior to the analysis. The dataset does not contain any missing values.

In [78]:
df.describe()

,Passport Number,Date of Visit,Sales Amount,Transaction Number
count,1.809500e+04,18095,18095.000000,18095.000000
mean,1.060900e+13,2013-03-25 20:22:01.823708160,24.041849,38074.109201
min,1.060900e+13,2012-09-19 00:00:00,0.000000,10001.000000
25%,1.060900e+13,2012-12-22 00:00:00,10.500000,20022.000000
50%,1.060900e+13,2013-03-15 00:00:00,17.500000,30168.000000
75%,1.060900e+13,2013-06-24 00:00:00,31.230000,50245.000000
max,1.060900e+13,2013-11-05 00:00:00,962.550000,130136.000000
std,3.592375e+05,NaN,26.560563,21833.069105


Through descriptive statistical analysis using describe(), we obtain an overall understanding of the data distribution. The dataset contains 18,095 transactions, while the recorded dates range from September 2012 to November 2013.

In addition, substantial variation is observed in purchase amounts, indicating differences in customer purchasing behavior.

In [80]:
df.head(2)

,Passport Number,Date of Visit,Sales Amount,Transaction Number
0,10609000872612,2013-11-05,12.20,10001
1,10609000885845,2012-12-11,962.55,10001


In [81]:
RFM = df.groupby('Passport Number').agg({'Date of Visit' : 'max',
                                         'Transaction Number' : 'count',
                                         'Sales Amount' : 'sum'
                                        })
RFM

,Date of Visit,Transaction Number,Sales Amount
Passport Number,,,
10609000000009,2013-08-05,1,13.38
10609000000014,2013-09-14,1,15.45
10609000000024,2013-09-20,1,15.03
10609000000027,2013-06-23,1,43.00
10609000001304,2012-10-15,2,25.30
...,...,...,...
10609001872155,2013-10-27,6,188.95
10609001872239,2013-04-22,5,124.20
10609001872312,2013-08-18,5,203.05


In the original dataset, each row represents an individual transaction. However, RFM analysis is performed at the customer level rather than at the transaction level. For this reason, the groupby() method is applied using Passport Number so that all transactions of each customer are aggregated into a single record.

The max() function is applied to the Date of Visit variable in order to identify the most recent visit date of each customer. This information is used to calculate Recency, which reflects how recently a customer interacted with the business.

The count() function is applied to the Transaction Number variable in order to calculate the number of transactions. This information is used to calculate Frequency, which reflects how often a customer makes purchases from the business.

Finally, the sum() function is applied to the Sales Amount variable in order to calculate the total monetary value generated by each customer. This information is used to calculate Monetary, which reflects the total economic value of each customer to the business.

In [83]:
RFM = RFM.rename(columns ={'Date of Visit' : 'Recency',
                     'Transaction Number' : 'Frequency',
                     'Sales Amount' : 'Monetary'
                    })

The columns are also renamed in order to improve clarity and readability.

In [85]:
RFM.head()

,Recency,Frequency,Monetary
Passport Number,,,
10609000000009,2013-08-05,1,13.38
10609000000014,2013-09-14,1,15.45
10609000000024,2013-09-20,1,15.03
10609000000027,2013-06-23,1,43.00
10609000001304,2012-10-15,2,25.30


In [86]:
RFM['R_score'] = 1
RFM.loc[RFM['Recency'] >= '2013-05-01', 'R_score'] = 2
RFM.loc[RFM['Recency'] >= '2013-09-01', 'R_score'] = 3
RFM

,Recency,Frequency,Monetary,R_score
Passport Number,,,,
10609000000009,2013-08-05,1,13.38,2
10609000000014,2013-09-14,1,15.45,3
10609000000024,2013-09-20,1,15.03,3
10609000000027,2013-06-23,1,43.00,2
10609000001304,2012-10-15,2,25.30,1
...,...,...,...,...
10609001872155,2013-10-27,6,188.95,3
10609001872239,2013-04-22,5,124.20,1
10609001872312,2013-08-18,5,203.05,2


Since the original variables are not easily utilized in segmentation, optimization, and marketing targeting processes, they are transformed into score classes. This process is based on the cut-off points provided in the assignment. Through this approach, customers are categorized into groups with similar characteristics, facilitating both comparison and business decision-making.

Initially, all customers receive a Recency score equal to 1, representing the lowest level of recent interaction with the business. Customers who visited after 01/05/2013 receive a score of 2, while customers who visited after 01/09/2013 receive a score of 3.

In [88]:
RFM['F_score'] = 1
RFM.loc[RFM['Frequency'] >= 10, 'F_score'] = 2
RFM.loc[RFM['Frequency'] >= 20, 'F_score'] = 3
RFM

,Recency,Frequency,Monetary,R_score,F_score
Passport Number,,,,,
10609000000009,2013-08-05,1,13.38,2,1
10609000000014,2013-09-14,1,15.45,3,1
10609000000024,2013-09-20,1,15.03,3,1
10609000000027,2013-06-23,1,43.00,2,1
10609000001304,2012-10-15,2,25.30,1,1
...,...,...,...,...,...
10609001872155,2013-10-27,6,188.95,3,1
10609001872239,2013-04-22,5,124.20,1,1
10609001872312,2013-08-18,5,203.05,2,1


A similar process is followed for customer categorization based on the Frequency score. Initially, all customers receive a score of 1, while customers with at least 10 transactions receive a score of 2 and customers with at least 20 transactions receive a score of 3.

This approach allows customers to be segmented according to the frequency of their visits and purchases.

In [90]:
RFM['M_score'] = 1
RFM.loc[RFM['Monetary'] >= 20, 'M_score'] = 2
RFM.loc[RFM['Monetary'] >= 40, 'M_score'] = 3
RFM

,Recency,Frequency,Monetary,R_score,F_score,M_score
Passport Number,,,,,,
10609000000009,2013-08-05,1,13.38,2,1,1
10609000000014,2013-09-14,1,15.45,3,1,1
10609000000024,2013-09-20,1,15.03,3,1,1
10609000000027,2013-06-23,1,43.00,2,1,3
10609000001304,2012-10-15,2,25.30,1,1,2
...,...,...,...,...,...,...
10609001872155,2013-10-27,6,188.95,3,1,3
10609001872239,2013-04-22,5,124.20,1,1,3
10609001872312,2013-08-18,5,203.05,2,1,3


Finally, customers are categorized based on the Monetary score. Initially, all customers receive a score of 1, while customers with total purchases above 20 units receive a score of 2 and customers with total purchases above 40 units receive a score of 3.

This process enables the segmentation of customers according to the overall economic value they provide to the business.

From a business perspective, customers with higher Frequency scores are considered more active and loyal to the business, increasing the likelihood of responding positively to future marketing campaigns.

In [92]:
RFM['Expected_profit'] = (0.45 * RFM['Monetary']) - 5.75
RFM

,Recency,Frequency,Monetary,R_score,F_score,M_score,Expected_profit
Passport Number,,,,,,,
10609000000009,2013-08-05,1,13.38,2,1,1,0.2710
10609000000014,2013-09-14,1,15.45,3,1,1,1.2025
10609000000024,2013-09-20,1,15.03,3,1,1,1.0135
10609000000027,2013-06-23,1,43.00,2,1,3,13.6000
10609000001304,2012-10-15,2,25.30,1,1,2,5.6350
...,...,...,...,...,...,...,...
10609001872155,2013-10-27,6,188.95,3,1,3,79.2775
10609001872239,2013-04-22,5,124.20,1,1,3,50.1400
10609001872312,2013-08-18,5,203.05,2,1,3,85.6225


According to the assignment, the probability of customer response to the marketing campaign can be either 25% or 65%, with equal probability of occurrence. For this reason, the average expected response probability is used:

(0.25 + 0.65) / 2 = 0.45

The Expected Profit for each customer is then calculated by subtracting the average campaign incentive cost from the expected revenue. According to the assignment, the average campaign cost amounts to $5.75 per customer.

This approach estimates the expected economic benefit that each customer may generate within the context of the upcoming marketing campaign.

In [94]:
RFM = RFM.reset_index()

Before grouping the data, reset_index() is performed so that Passport Number becomes a regular dataframe column again and can be used in the aggregation process.

In [96]:
RFM.groupby('R_score').agg({'Passport Number': 'count',
                            'Expected_profit': 'sum'
                           })

,Passport Number,Expected_profit
R_score,,
1,1495,23197.9915
2,323,25255.3715
3,502,133973.3995


Grouping by R_score allows the evaluation of customer profitability based on how recently customers visited the restaurant. For each category, both the total expected profit and the number of customers are calculated.

The results clearly indicate that customers with R_score = 3 represent the most valuable group for the upcoming campaign, as they generate significantly higher expected profit compared to the remaining categories. This finding supports the assumption that customers with more recent interaction with the business are more likely to respond positively to future promotional activities.

In [98]:
RFM.groupby('F_score').agg({'Passport Number': 'count',
                            'Expected_profit': 'sum'
                           })

,Passport Number,Expected_profit
F_score,,
1,1971,49713.1245
2,172,26003.5795
3,177,106710.0585


The analysis of F_score focuses on customer visit frequency and the extent to which it affects the expected profitability of the campaign. For this purpose, both the number of customers per category and the total expected profit generated by each group are examined.

Although customers with F_score = 3 represent a smaller portion of the total customer base, they contribute disproportionately to the total expected profit. This suggests that highly frequent customers have developed a stronger relationship with the business and therefore represent an especially attractive target for future marketing campaigns.

In [100]:
RFM.groupby('M_score').agg({'Passport Number': 'count',
                            'Expected_profit': 'sum'
                           })

,Passport Number,Expected_profit
M_score,,
1,539,-583.1785
2,464,3616.0115
3,1317,179393.9295


The examination of M_score highlights the differences in the overall economic value of customers. By grouping customers according to their Monetary score class, it becomes possible to evaluate the expected profit generated by each category within the context of the marketing campaign.

Customers with M_score = 3 account for the majority of the total expected profit, making them the most profitable group for campaign targeting. In contrast, the M_score = 1 category generates negative expected profit, indicating that the campaign participation cost exceeds the expected economic benefit.

Therefore, under budget constraints, this specific customer segment would not be considered strategically efficient for campaign targeting.

In [102]:
# summary

Overall, the RFM and Expected Profit analysis demonstrates that customers with high Recency, Frequency, and Monetary scores represent the most profitable groups for targeting within the upcoming marketing campaign.

In particular, customers with high economic value and frequent interaction with the business generate disproportionately higher expected profit compared to the rest of the customer base.

At the same time, limited campaign budget makes customer prioritization essential, since certain categories, such as customers with low Monetary scores, generate very low or even negative expected profitability.

Therefore, the application of the RFM methodology combined with Expected Profit analysis can significantly support customer segmentation and improve the efficiency of marketing targeting strategies.

In [ ]:
RFM

,Passport Number,Recency,Frequency,Monetary,R_score,F_score,M_score,Expected_profit
0,10609000000009,2013-08-05,1,13.38,2,1,1,0.2710
1,10609000000014,2013-09-14,1,15.45,3,1,1,1.2025
2,10609000000024,2013-09-20,1,15.03,3,1,1,1.0135
3,10609000000027,2013-06-23,1,43.00,2,1,3,13.6000
4,10609000001304,2012-10-15,2,25.30,1,1,2,5.6350
...,...,...,...,...,...,...,...,...
2315,10609001872155,2013-10-27,6,188.95,3,1,3,79.2775
2316,10609001872239,2013-04-22,5,124.20,1,1,3,50.1400
2317,10609001872312,2013-08-18,5,203.05,2,1,3,85.6225
2318,10609001872494,2013-07-02,2,117.30,2,1,3,47.0350
